In [ ]:
import numpy
import pandas
import uproot

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

import sbruceana

In [ ]:
PATH_TO_SBRUCE_NOM = "/Users/triozzi/Analysis/numine/sbruceana/data/cc1mu0pi/cv/"
FILE_CV = "CNAF_CV_1muNp0pi_NuMI_NoSysts_5400.root"
FILE_VAR1 = "CNAF_CV_1muNp0pi_NuMI_NoSysts_var1_5400.root"

DET_VARS = {
  'CV'            : FILE_CV,
  'hi coh noise'  : FILE_VAR1,
}

BINNING = numpy.array([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1., 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.75, 2, 2.5, 3])
# BINNING = numpy.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 1.75, 2, 2.5, 3])

In [ ]:
dfs, pots, lives, tags = [], [], [], []

for tag, file in DET_VARS.items():

  # get sbruce tree
  df = sbruceana.io.convert_tree_to_df(
    f"{PATH_TO_SBRUCE_NOM}{file}",
    "events/selectedNu"  
  )
  dfs.append(df)

  # get POTs
  pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE_NOM}{file}")
  pots.append(pot)

  # get livetimes
  live = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE_NOM}{file}")
  lives.append(live)

  # get variation tag
  tags.append(tag)

dfs = numpy.array(dfs, dtype=object)
pots = numpy.array(pots)
lives = numpy.array(lives)
tags = numpy.array(tags)

In [ ]:
pots, lives

In [ ]:
fig, ax = plt.subplots(figsize=(4,3), layout='constrained')

var = "recoE"

# width = 0.2; bins = numpy.arange(0.2, 3+width, width)
bins = BINNING

ax = sbruceana.plotting.plot_var_by_category(ax, dfs[0], bins, var, 'numu', True)

# gfx
ax.set(
  title  = f'CV vs variations',
  xlabel = f'{var} [GeV]',
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig("plots/CNAF_NuE_1eNp0pi_NuMI.png", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3), layout='constrained')

var = "recoE"

# width = 0.2; bins = numpy.arange(0.2, 3+width, width)
bins = BINNING

ax = sbruceana.plotting.plot_var_by_category(ax, dfs[0], bins, var, 'numu', True, hatch='//')

for df, pot, tag in zip(dfs[1:], pots[1:], tags[1:]):
  ax = sbruceana.plotting.plot_var(ax, df, bins, var, tag, pots[0] / pot)

# gfx
ax.set(
  title  = f'CV vs variations',
  xlabel = f'{var} [GeV]',
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig("plots/CNAF_NuE_1eNp0pi_NuMI_DetVars.png", dpi=300)

In [ ]:
fig, (ax, ax_ratio) = plt.subplots(
  2, 1,
  figsize     = (4.5, 4),
  gridspec_kw = {
    'height_ratios': [2, 2], 
    'hspace': 0.03
  },
  sharex      = True,
  layout      = 'constrained'
)

var = "recoE"

bins = BINNING
x = 0.5 * (bins[:-1] + bins[1:])

# CV
ax = sbruceana.plotting.plot_var_by_category(ax, dfs[0], bins, var)

# variations
i = 0
for df, pot, tag in zip(dfs[1:], pots[1:], tags[1:]):
  ax = sbruceana.plotting.plot_var(ax, df, bins, var, tag, pots[0] / pot)

  # residuals
  r, er = sbruceana.utils.get_ratio_of_vars(dfs[0][var], pots[0], df[var], pot, bins)
  ax_ratio.errorbar(x, r, yerr=er, fmt=f'C{i+2}', marker='.', markersize=6, linewidth=1.25)
  i += 1

# references
ax_ratio.axhline(1, c='black', lw=0.75)
cv,  _ = numpy.histogram(dfs[0][var], bins=bins)
cv_err_ratio = numpy.where(cv > 0, numpy.sqrt(cv) / cv, numpy.nan)
ax_ratio.bar(x, 2*cv_err_ratio, width = numpy.diff(bins), bottom = 1-cv_err_ratio, color='gray', alpha=0.3, fill=True, lw=0)

# gfx
ax.set(
  title  = f'CV vs variations',
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(ncols=2, loc='upper right', title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(11)
ax_ratio.set(
  xlabel = f'{var} [GeV]',
  ylabel = 'var / CV',
)

fig.savefig("plots/CNAF_NuE_1eNp0pi_NuMI_DetVars_WithRes.png", dpi=300)

In [ ]:
fig, (ax, ax_ratio) = plt.subplots(
  2, 1,
  figsize     = (5, 4),
  gridspec_kw = {
    'height_ratios': [2., 1], 
    'hspace': 0.025
  },
  sharex      = True,
  layout      = 'constrained'
)

var = "recoE"

bins = BINNING
x = 0.5 * (bins[:-1] + bins[1:])

# CV
ax = sbruceana.plotting.plot_var_by_category(ax, dfs[0], bins, var)

# variations
DET_VAR_IDX = [1]
DET_VAR_TAG = 'coh noise'
i = 0
for df, pot, tag in zip(dfs[DET_VAR_IDX], pots[DET_VAR_IDX], tags[DET_VAR_IDX]):
  if len(DET_VAR_IDX) == 1:
    ax = sbruceana.plotting.plot_var(ax, df, bins, var, tag, pots[0] / pot, color='C3')
  else:
    ax = sbruceana.plotting.plot_var(ax, df, bins, var, tag, pots[0] / pot)

  # residuals
  r, er = sbruceana.utils.get_ratio_of_vars(dfs[0][var], pots[0], df[var], pot, bins)

  if tag == 'null':
    ax_ratio.plot(x, r, color=f'C{i+2}', linewidth=1.25)
    ax_ratio.fill_between(x, r - er, r + er, color=f'C{i+2}', alpha=0.2, ec=None)
  else:
    if len(DET_VAR_IDX) == 1:
      this_color = f'C3'
    else:
      this_color = f'C{i+2}'
    ax_ratio.errorbar(x, r, xerr=numpy.diff(bins)/2, yerr=er, fmt=this_color, ls='', marker='', markersize=6, linewidth=1.5)
  i += 1

# references
ax_ratio.axhline(1, c='black', lw=0.75)
cv,  _ = numpy.histogram(dfs[0][var], bins=bins)
cv_err_ratio = numpy.where(cv > 0, numpy.sqrt(cv) / cv, numpy.nan)
ax_ratio.bar(x, 2*cv_err_ratio, width = numpy.diff(bins), bottom = 1-cv_err_ratio, color='gray', alpha=0.3, fill=True, lw=0)

# gfx
ax.set(
  title  = f'CV vs {DET_VAR_TAG} variation',
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(ncols=2, loc='upper right', fontsize=9.5, title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(10)
ax_ratio.set(
  xlabel = 'reconstructed $E_{\\nu}$ [GeV]',
  ylabel = 'var / CV',
  ylim   = (0.8, 1.2),
)

PLOT_NAME = f"plots/detector/CNAF_NuE_1eNp0pi_NuMI_DetVars_WithRes_{DET_VAR_TAG.replace(' ', '_')}"
fig.savefig(f"{PLOT_NAME}.png", dpi=300); fig.savefig(f"{PLOT_NAME}.pdf", dpi=300)